# Bayesian Regression Analysis: Computational Model Parameters and Mental Health Scores

This notebook performs Bayesian regression analyses using Bambi to model relationships between computational model parameters (z, k, beta) and mental health questionnaire scores.

## Analysis Overview
- **Parameters**: z (starting point), k (learning rate), beta (inverse temperature)
- **Mental Health Measures**: 
  - DASS21 (Depression, Anxiety, Stress Scale) - Total and Subscales
  - AMI (Apathy Motivation Index) - Total and Subscales
  - MFIS (Modified Fatigue Impact Scale) - Total and Subscales
  - State Anxiety (STAI-State)
  - Trait Anxiety (STAI-Trait)
  - OASIS (Overall Anxiety Severity and Impairment Scale)
  - PHQ-9 (Patient Health Questionnaire-9)
  
We will examine both total scores and subscale-level relationships.

## Setup and Data Loading

In [ ]:
# Install required packages if needed
# !pip install bambi arviz pandas numpy matplotlib seaborn --break-system-packages

In [ ]:
import pandas as pd
import numpy as np
import bambi as bmb
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
az.style.use('arviz-darkgrid')

# Set random seed for reproducibility
RANDOM_SEED = 42

### Load Parameter Files

In [ ]:
# Load computational model parameters
beta_params = pd.read_csv('FET_Exp_Bias_beta_params.csv')
k_params = pd.read_csv('FET_Exp_Bias_k_params.csv')
z_params = pd.read_csv('FET_Exp_Bias_z_params.csv')

# Rename columns for clarity
beta_params = beta_params.rename(columns={'mean': 'beta_mean', 'std': 'beta_std'})
k_params = k_params.rename(columns={'mean': 'k_mean', 'std': 'k_std'})
z_params = z_params.rename(columns={'mean': 'z_mean', 'std': 'z_std'})

# Display shapes
print(f"Beta params shape: {beta_params.shape}")
print(f"K params shape: {k_params.shape}")
print(f"Z params shape: {z_params.shape}")

### Load Mental Health Questionnaire Data

In [ ]:
# Load mental health questionnaire data
psych_data = pd.read_csv('psych_270.csv')

print(f"Psych data shape: {psych_data.shape}")
print(f"\nColumns: {psych_data.columns.tolist()}")

### Merge and Prepare Data

In [ ]:
# Merge all parameter data
params = beta_params[['subject', 'beta_mean', 'beta_std']].merge(
    k_params[['subject', 'k_mean', 'k_std']], on='subject'
).merge(
    z_params[['subject', 'z_mean', 'z_std']], on='subject'
)

# Rename psych subject column to match
psych_data = psych_data.rename(columns={'subj': 'subject'})

# Merge with psych data
df = params.merge(psych_data, on='subject', how='inner')

print(f"Merged data shape: {df.shape}")
print(f"Number of subjects: {df['subject'].nunique()}")

### Compute Total Scores

In [ ]:
# Compute DASS21 Total Score (sum of Depression, Anxiety, Stress)
df['DASS21_Total'] = df['DASS21_Depression_Score'] + df['DASS21_Anxiety_Score'] + df['DASS21_Stress_Score']

# Compute AMI Total Score (sum of Behavioural, Social, Emotional)
df['AMI_Total'] = df['AMI_Behavioural_Score'] + df['AMI_Social_Score'] + df['AMI_Emotional_Score']

# Compute MFIS Total Score (sum of Physical, Cognitive, Psychosocial)
df['MFIS_Total'] = df['MFIS_Physical_Score'] + df['MFIS_Cognitive_Score'] + df['MFIS_Psychosocial_Score']

print("Total scores computed:")
print(f"  DASS21_Total: {df['DASS21_Total'].describe().round(2).to_dict()}")
print(f"  AMI_Total: {df['AMI_Total'].describe().round(2).to_dict()}")
print(f"  MFIS_Total: {df['MFIS_Total'].describe().round(2).to_dict()}")
print(f"  Total_State_Score: {df['Total_State_Score'].describe().round(2).to_dict()}")
print(f"  Total_Trait_Score: {df['Total_Trait_Score'].describe().round(2).to_dict()}")
print(f"  Total_OASIS_Score: {df['Total_OASIS_Score'].describe().round(2).to_dict()}")
print(f"  Total_PHQ9_Score: {df['Total_PHQ9_Score'].describe().round(2).to_dict()}")

In [ ]:
# Check for missing values in key variables
key_vars = ['beta_mean', 'k_mean', 'z_mean', 
            'DASS21_Total', 'AMI_Total', 'MFIS_Total',
            'DASS21_Depression_Score', 'DASS21_Anxiety_Score', 'DASS21_Stress_Score',
            'AMI_Behavioural_Score', 'AMI_Social_Score', 'AMI_Emotional_Score',
            'MFIS_Physical_Score', 'MFIS_Cognitive_Score', 'MFIS_Psychosocial_Score',
            'Total_State_Score', 'Total_Trait_Score', 'Total_OASIS_Score', 'Total_PHQ9_Score']

missing_summary = df[key_vars].isnull().sum()
print("Missing values per variable:")
print(missing_summary[missing_summary > 0])

# Drop rows with missing values for analysis
df_clean = df.dropna(subset=key_vars)
print(f"\nSubjects after removing missing values: {len(df_clean)}")

### Standardize Variables for Better Model Convergence

In [ ]:
# Standardize predictors (mental health scores) and outcomes (parameters)
from scipy import stats

# Parameters to model
params_to_model = ['beta_mean', 'k_mean', 'z_mean']

# Mental health scores - computed totals
computed_totals = ['DASS21_Total', 'AMI_Total', 'MFIS_Total']

# Mental health scores - direct totals from questionnaires
direct_totals = ['Total_State_Score', 'Total_Trait_Score', 'Total_OASIS_Score', 'Total_PHQ9_Score']

# All total scores
all_total_scores = computed_totals + direct_totals

# Subscales
dass_subscales = ['DASS21_Depression_Score', 'DASS21_Anxiety_Score', 'DASS21_Stress_Score']
ami_subscales = ['AMI_Behavioural_Score', 'AMI_Social_Score', 'AMI_Emotional_Score']
mfis_subscales = ['MFIS_Physical_Score', 'MFIS_Cognitive_Score', 'MFIS_Psychosocial_Score']

all_predictors = all_total_scores + dass_subscales + ami_subscales + mfis_subscales

# Create standardized versions
for var in params_to_model + all_predictors:
    df_clean[f'{var}_z'] = stats.zscore(df_clean[var], nan_policy='omit')

print("Variables standardized successfully.")
print(f"\nTotal scores available: {all_total_scores}")

## Exploratory Data Analysis

In [ ]:
# Distribution of parameters
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, param in zip(axes, params_to_model):
    ax.hist(df_clean[param], bins=30, edgecolor='black', alpha=0.7)
    ax.set_xlabel(param)
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {param}')

fig.tight_layout()
plt.show()

In [ ]:
# Distribution of all mental health total scores
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, score in enumerate(all_total_scores):
    ax = axes[idx]
    ax.hist(df_clean[score], bins=30, edgecolor='black', alpha=0.7)
    ax.set_xlabel(score)
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {score}')

# Hide the last empty subplot
axes[-1].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
# Correlation matrix - all total scores
correlation_vars = params_to_model + all_total_scores
corr_matrix = df_clean[correlation_vars].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix: Parameters and All Mental Health Total Scores')
plt.show()

---
# Helper Functions for Model Fitting and Summary

In [ ]:
def fit_bayesian_model(formula, data, model_name="model", draws=2000, tune=1000, cores=2):
    """
    Fit a Bayesian regression model using Bambi.
    
    Parameters:
    -----------
    formula : str
        Bambi/patsy style formula
    data : pd.DataFrame
        Data for the model
    model_name : str
        Name for the model
    draws : int
        Number of posterior draws per chain
    tune : int
        Number of tuning samples
    cores : int
        Number of CPU cores for parallel sampling
    
    Returns:
    --------
    model, idata : tuple
        Fitted model and inference data
    """
    print(f"\n{'='*60}")
    print(f"Fitting Model: {model_name}")
    print(f"Formula: {formula}")
    print(f"{'='*60}")
    
    model = bmb.Model(formula, data)
    # Include log_likelihood for model comparison (LOO-CV)
    idata = model.fit(draws=draws, tune=tune, cores=cores, random_seed=RANDOM_SEED,
                      idata_kwargs={"log_likelihood": True})
    
    return model, idata


def summarize_model(model, idata, model_name="model"):
    """
    Summarize and visualize model results.
    """
    print(f"\n--- Model Summary: {model_name} ---\n")
    
    # Summary statistics
    summary = az.summary(idata, hdi_prob=0.95)
    print(summary)
    
    return summary


def plot_posterior(idata, var_names=None, model_name="model"):
    """
    Plot posterior distributions.
    """
    az.plot_posterior(idata, var_names=var_names, hdi_prob=0.95)
    plt.suptitle(f'Posterior Distributions: {model_name}', y=1.02)
    plt.show()


def plot_forest(idata, var_names=None, model_name="model"):
    """
    Forest plot of coefficients.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    az.plot_forest(idata, var_names=var_names, hdi_prob=0.95, combined=True, ax=ax)
    ax.set_title(f'Forest Plot: {model_name}')
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)
    plt.show()

---
# Part 1: Total Score Models (Individual Predictors)

We will model each parameter (beta, k, z) as predicted by each total score individually.

## 1.1 Beta ~ Individual Total Scores

In [ ]:
# Store all idata objects for later comparison
beta_idatas = {}
beta_models = {}

In [ ]:
# Model: beta ~ DASS21_Total
beta_models['DASS21'], beta_idatas['DASS21'] = fit_bayesian_model(
    formula="beta_mean_z ~ DASS21_Total_z",
    data=df_clean,
    model_name="Beta ~ DASS21 Total"
)
summarize_model(beta_models['DASS21'], beta_idatas['DASS21'], "Beta ~ DASS21 Total")
plot_forest(beta_idatas['DASS21'], var_names=['DASS21_Total_z'], model_name="Beta ~ DASS21 Total")

In [ ]:
# Model: beta ~ AMI_Total
beta_models['AMI'], beta_idatas['AMI'] = fit_bayesian_model(
    formula="beta_mean_z ~ AMI_Total_z",
    data=df_clean,
    model_name="Beta ~ AMI Total"
)
summarize_model(beta_models['AMI'], beta_idatas['AMI'], "Beta ~ AMI Total")
plot_forest(beta_idatas['AMI'], var_names=['AMI_Total_z'], model_name="Beta ~ AMI Total")

In [ ]:
# Model: beta ~ MFIS_Total
beta_models['MFIS'], beta_idatas['MFIS'] = fit_bayesian_model(
    formula="beta_mean_z ~ MFIS_Total_z",
    data=df_clean,
    model_name="Beta ~ MFIS Total"
)
summarize_model(beta_models['MFIS'], beta_idatas['MFIS'], "Beta ~ MFIS Total")
plot_forest(beta_idatas['MFIS'], var_names=['MFIS_Total_z'], model_name="Beta ~ MFIS Total")

In [ ]:
# Model: beta ~ Total_State_Score
beta_models['State'], beta_idatas['State'] = fit_bayesian_model(
    formula="beta_mean_z ~ Total_State_Score_z",
    data=df_clean,
    model_name="Beta ~ State Anxiety"
)
summarize_model(beta_models['State'], beta_idatas['State'], "Beta ~ State Anxiety")
plot_forest(beta_idatas['State'], var_names=['Total_State_Score_z'], model_name="Beta ~ State Anxiety")

In [ ]:
# Model: beta ~ Total_Trait_Score
beta_models['Trait'], beta_idatas['Trait'] = fit_bayesian_model(
    formula="beta_mean_z ~ Total_Trait_Score_z",
    data=df_clean,
    model_name="Beta ~ Trait Anxiety"
)
summarize_model(beta_models['Trait'], beta_idatas['Trait'], "Beta ~ Trait Anxiety")
plot_forest(beta_idatas['Trait'], var_names=['Total_Trait_Score_z'], model_name="Beta ~ Trait Anxiety")

In [ ]:
# Model: beta ~ Total_OASIS_Score
beta_models['OASIS'], beta_idatas['OASIS'] = fit_bayesian_model(
    formula="beta_mean_z ~ Total_OASIS_Score_z",
    data=df_clean,
    model_name="Beta ~ OASIS"
)
summarize_model(beta_models['OASIS'], beta_idatas['OASIS'], "Beta ~ OASIS")
plot_forest(beta_idatas['OASIS'], var_names=['Total_OASIS_Score_z'], model_name="Beta ~ OASIS")

In [ ]:
# Model: beta ~ Total_PHQ9_Score
beta_models['PHQ9'], beta_idatas['PHQ9'] = fit_bayesian_model(
    formula="beta_mean_z ~ Total_PHQ9_Score_z",
    data=df_clean,
    model_name="Beta ~ PHQ9"
)
summarize_model(beta_models['PHQ9'], beta_idatas['PHQ9'], "Beta ~ PHQ9")
plot_forest(beta_idatas['PHQ9'], var_names=['Total_PHQ9_Score_z'], model_name="Beta ~ PHQ9")

## 1.2 K ~ Individual Total Scores

In [ ]:
# Store all idata objects for later comparison
k_idatas = {}
k_models = {}

In [ ]:
# Model: k ~ DASS21_Total
k_models['DASS21'], k_idatas['DASS21'] = fit_bayesian_model(
    formula="k_mean_z ~ DASS21_Total_z",
    data=df_clean,
    model_name="K ~ DASS21 Total"
)
summarize_model(k_models['DASS21'], k_idatas['DASS21'], "K ~ DASS21 Total")
plot_forest(k_idatas['DASS21'], var_names=['DASS21_Total_z'], model_name="K ~ DASS21 Total")

In [ ]:
# Model: k ~ AMI_Total
k_models['AMI'], k_idatas['AMI'] = fit_bayesian_model(
    formula="k_mean_z ~ AMI_Total_z",
    data=df_clean,
    model_name="K ~ AMI Total"
)
summarize_model(k_models['AMI'], k_idatas['AMI'], "K ~ AMI Total")
plot_forest(k_idatas['AMI'], var_names=['AMI_Total_z'], model_name="K ~ AMI Total")

In [ ]:
# Model: k ~ MFIS_Total
k_models['MFIS'], k_idatas['MFIS'] = fit_bayesian_model(
    formula="k_mean_z ~ MFIS_Total_z",
    data=df_clean,
    model_name="K ~ MFIS Total"
)
summarize_model(k_models['MFIS'], k_idatas['MFIS'], "K ~ MFIS Total")
plot_forest(k_idatas['MFIS'], var_names=['MFIS_Total_z'], model_name="K ~ MFIS Total")

In [ ]:
# Model: k ~ Total_State_Score
k_models['State'], k_idatas['State'] = fit_bayesian_model(
    formula="k_mean_z ~ Total_State_Score_z",
    data=df_clean,
    model_name="K ~ State Anxiety"
)
summarize_model(k_models['State'], k_idatas['State'], "K ~ State Anxiety")
plot_forest(k_idatas['State'], var_names=['Total_State_Score_z'], model_name="K ~ State Anxiety")

In [ ]:
# Model: k ~ Total_Trait_Score
k_models['Trait'], k_idatas['Trait'] = fit_bayesian_model(
    formula="k_mean_z ~ Total_Trait_Score_z",
    data=df_clean,
    model_name="K ~ Trait Anxiety"
)
summarize_model(k_models['Trait'], k_idatas['Trait'], "K ~ Trait Anxiety")
plot_forest(k_idatas['Trait'], var_names=['Total_Trait_Score_z'], model_name="K ~ Trait Anxiety")

In [ ]:
# Model: k ~ Total_OASIS_Score
k_models['OASIS'], k_idatas['OASIS'] = fit_bayesian_model(
    formula="k_mean_z ~ Total_OASIS_Score_z",
    data=df_clean,
    model_name="K ~ OASIS"
)
summarize_model(k_models['OASIS'], k_idatas['OASIS'], "K ~ OASIS")
plot_forest(k_idatas['OASIS'], var_names=['Total_OASIS_Score_z'], model_name="K ~ OASIS")

In [ ]:
# Model: k ~ Total_PHQ9_Score
k_models['PHQ9'], k_idatas['PHQ9'] = fit_bayesian_model(
    formula="k_mean_z ~ Total_PHQ9_Score_z",
    data=df_clean,
    model_name="K ~ PHQ9"
)
summarize_model(k_models['PHQ9'], k_idatas['PHQ9'], "K ~ PHQ9")
plot_forest(k_idatas['PHQ9'], var_names=['Total_PHQ9_Score_z'], model_name="K ~ PHQ9")

## 1.3 Z ~ Individual Total Scores

In [ ]:
# Store all idata objects for later comparison
z_idatas = {}
z_models_dict = {}

In [ ]:
# Model: z ~ DASS21_Total
z_models_dict['DASS21'], z_idatas['DASS21'] = fit_bayesian_model(
    formula="z_mean_z ~ DASS21_Total_z",
    data=df_clean,
    model_name="Z ~ DASS21 Total"
)
summarize_model(z_models_dict['DASS21'], z_idatas['DASS21'], "Z ~ DASS21 Total")
plot_forest(z_idatas['DASS21'], var_names=['DASS21_Total_z'], model_name="Z ~ DASS21 Total")

In [ ]:
# Model: z ~ AMI_Total
z_models_dict['AMI'], z_idatas['AMI'] = fit_bayesian_model(
    formula="z_mean_z ~ AMI_Total_z",
    data=df_clean,
    model_name="Z ~ AMI Total"
)
summarize_model(z_models_dict['AMI'], z_idatas['AMI'], "Z ~ AMI Total")
plot_forest(z_idatas['AMI'], var_names=['AMI_Total_z'], model_name="Z ~ AMI Total")

In [ ]:
# Model: z ~ MFIS_Total
z_models_dict['MFIS'], z_idatas['MFIS'] = fit_bayesian_model(
    formula="z_mean_z ~ MFIS_Total_z",
    data=df_clean,
    model_name="Z ~ MFIS Total"
)
summarize_model(z_models_dict['MFIS'], z_idatas['MFIS'], "Z ~ MFIS Total")
plot_forest(z_idatas['MFIS'], var_names=['MFIS_Total_z'], model_name="Z ~ MFIS Total")

In [ ]:
# Model: z ~ Total_State_Score
z_models_dict['State'], z_idatas['State'] = fit_bayesian_model(
    formula="z_mean_z ~ Total_State_Score_z",
    data=df_clean,
    model_name="Z ~ State Anxiety"
)
summarize_model(z_models_dict['State'], z_idatas['State'], "Z ~ State Anxiety")
plot_forest(z_idatas['State'], var_names=['Total_State_Score_z'], model_name="Z ~ State Anxiety")

In [ ]:
# Model: z ~ Total_Trait_Score
z_models_dict['Trait'], z_idatas['Trait'] = fit_bayesian_model(
    formula="z_mean_z ~ Total_Trait_Score_z",
    data=df_clean,
    model_name="Z ~ Trait Anxiety"
)
summarize_model(z_models_dict['Trait'], z_idatas['Trait'], "Z ~ Trait Anxiety")
plot_forest(z_idatas['Trait'], var_names=['Total_Trait_Score_z'], model_name="Z ~ Trait Anxiety")

In [ ]:
# Model: z ~ Total_OASIS_Score
z_models_dict['OASIS'], z_idatas['OASIS'] = fit_bayesian_model(
    formula="z_mean_z ~ Total_OASIS_Score_z",
    data=df_clean,
    model_name="Z ~ OASIS"
)
summarize_model(z_models_dict['OASIS'], z_idatas['OASIS'], "Z ~ OASIS")
plot_forest(z_idatas['OASIS'], var_names=['Total_OASIS_Score_z'], model_name="Z ~ OASIS")

In [ ]:
# Model: z ~ Total_PHQ9_Score
z_models_dict['PHQ9'], z_idatas['PHQ9'] = fit_bayesian_model(
    formula="z_mean_z ~ Total_PHQ9_Score_z",
    data=df_clean,
    model_name="Z ~ PHQ9"
)
summarize_model(z_models_dict['PHQ9'], z_idatas['PHQ9'], "Z ~ PHQ9")
plot_forest(z_idatas['PHQ9'], var_names=['Total_PHQ9_Score_z'], model_name="Z ~ PHQ9")

---
# Part 2: Subscale Models

Examine relationships with individual subscales of DASS21, AMI, and MFIS.

## 2.1 Beta ~ Subscales

In [ ]:
# Model: beta ~ all DASS21 subscales
model_beta_dass_sub, idata_beta_dass_sub = fit_bayesian_model(
    formula="beta_mean_z ~ DASS21_Depression_Score_z + DASS21_Anxiety_Score_z + DASS21_Stress_Score_z",
    data=df_clean,
    model_name="Beta ~ DASS21 Subscales"
)
summarize_model(model_beta_dass_sub, idata_beta_dass_sub, "Beta ~ DASS21 Subscales")
plot_forest(idata_beta_dass_sub, 
            var_names=['DASS21_Depression_Score_z', 'DASS21_Anxiety_Score_z', 'DASS21_Stress_Score_z'],
            model_name="Beta ~ DASS21 Subscales")

In [ ]:
# Model: beta ~ all AMI subscales
model_beta_ami_sub, idata_beta_ami_sub = fit_bayesian_model(
    formula="beta_mean_z ~ AMI_Behavioural_Score_z + AMI_Social_Score_z + AMI_Emotional_Score_z",
    data=df_clean,
    model_name="Beta ~ AMI Subscales"
)
summarize_model(model_beta_ami_sub, idata_beta_ami_sub, "Beta ~ AMI Subscales")
plot_forest(idata_beta_ami_sub, 
            var_names=['AMI_Behavioural_Score_z', 'AMI_Social_Score_z', 'AMI_Emotional_Score_z'],
            model_name="Beta ~ AMI Subscales")

In [ ]:
# Model: beta ~ all MFIS subscales
model_beta_mfis_sub, idata_beta_mfis_sub = fit_bayesian_model(
    formula="beta_mean_z ~ MFIS_Physical_Score_z + MFIS_Cognitive_Score_z + MFIS_Psychosocial_Score_z",
    data=df_clean,
    model_name="Beta ~ MFIS Subscales"
)
summarize_model(model_beta_mfis_sub, idata_beta_mfis_sub, "Beta ~ MFIS Subscales")
plot_forest(idata_beta_mfis_sub, 
            var_names=['MFIS_Physical_Score_z', 'MFIS_Cognitive_Score_z', 'MFIS_Psychosocial_Score_z'],
            model_name="Beta ~ MFIS Subscales")

## 2.2 K ~ Subscales

In [ ]:
# Model: k ~ all DASS21 subscales
model_k_dass_sub, idata_k_dass_sub = fit_bayesian_model(
    formula="k_mean_z ~ DASS21_Depression_Score_z + DASS21_Anxiety_Score_z + DASS21_Stress_Score_z",
    data=df_clean,
    model_name="K ~ DASS21 Subscales"
)
summarize_model(model_k_dass_sub, idata_k_dass_sub, "K ~ DASS21 Subscales")
plot_forest(idata_k_dass_sub, 
            var_names=['DASS21_Depression_Score_z', 'DASS21_Anxiety_Score_z', 'DASS21_Stress_Score_z'],
            model_name="K ~ DASS21 Subscales")

In [ ]:
# Model: k ~ all AMI subscales
model_k_ami_sub, idata_k_ami_sub = fit_bayesian_model(
    formula="k_mean_z ~ AMI_Behavioural_Score_z + AMI_Social_Score_z + AMI_Emotional_Score_z",
    data=df_clean,
    model_name="K ~ AMI Subscales"
)
summarize_model(model_k_ami_sub, idata_k_ami_sub, "K ~ AMI Subscales")
plot_forest(idata_k_ami_sub, 
            var_names=['AMI_Behavioural_Score_z', 'AMI_Social_Score_z', 'AMI_Emotional_Score_z'],
            model_name="K ~ AMI Subscales")

In [ ]:
# Model: k ~ all MFIS subscales
model_k_mfis_sub, idata_k_mfis_sub = fit_bayesian_model(
    formula="k_mean_z ~ MFIS_Physical_Score_z + MFIS_Cognitive_Score_z + MFIS_Psychosocial_Score_z",
    data=df_clean,
    model_name="K ~ MFIS Subscales"
)
summarize_model(model_k_mfis_sub, idata_k_mfis_sub, "K ~ MFIS Subscales")
plot_forest(idata_k_mfis_sub, 
            var_names=['MFIS_Physical_Score_z', 'MFIS_Cognitive_Score_z', 'MFIS_Psychosocial_Score_z'],
            model_name="K ~ MFIS Subscales")

## 2.3 Z ~ Subscales

In [ ]:
# Model: z ~ all DASS21 subscales
model_z_dass_sub, idata_z_dass_sub = fit_bayesian_model(
    formula="z_mean_z ~ DASS21_Depression_Score_z + DASS21_Anxiety_Score_z + DASS21_Stress_Score_z",
    data=df_clean,
    model_name="Z ~ DASS21 Subscales"
)
summarize_model(model_z_dass_sub, idata_z_dass_sub, "Z ~ DASS21 Subscales")
plot_forest(idata_z_dass_sub, 
            var_names=['DASS21_Depression_Score_z', 'DASS21_Anxiety_Score_z', 'DASS21_Stress_Score_z'],
            model_name="Z ~ DASS21 Subscales")

In [ ]:
# Model: z ~ all AMI subscales
model_z_ami_sub, idata_z_ami_sub = fit_bayesian_model(
    formula="z_mean_z ~ AMI_Behavioural_Score_z + AMI_Social_Score_z + AMI_Emotional_Score_z",
    data=df_clean,
    model_name="Z ~ AMI Subscales"
)
summarize_model(model_z_ami_sub, idata_z_ami_sub, "Z ~ AMI Subscales")
plot_forest(idata_z_ami_sub, 
            var_names=['AMI_Behavioural_Score_z', 'AMI_Social_Score_z', 'AMI_Emotional_Score_z'],
            model_name="Z ~ AMI Subscales")

In [ ]:
# Model: z ~ all MFIS subscales
model_z_mfis_sub, idata_z_mfis_sub = fit_bayesian_model(
    formula="z_mean_z ~ MFIS_Physical_Score_z + MFIS_Cognitive_Score_z + MFIS_Psychosocial_Score_z",
    data=df_clean,
    model_name="Z ~ MFIS Subscales"
)
summarize_model(model_z_mfis_sub, idata_z_mfis_sub, "Z ~ MFIS Subscales")
plot_forest(idata_z_mfis_sub, 
            var_names=['MFIS_Physical_Score_z', 'MFIS_Cognitive_Score_z', 'MFIS_Psychosocial_Score_z'],
            model_name="Z ~ MFIS Subscales")

---
# Part 3: Combined Models

Models with multiple total scores as predictors.

In [ ]:
# Model: beta ~ all total scores
model_beta_all, idata_beta_all = fit_bayesian_model(
    formula="beta_mean_z ~ DASS21_Total_z + AMI_Total_z + MFIS_Total_z + Total_State_Score_z + Total_Trait_Score_z + Total_OASIS_Score_z + Total_PHQ9_Score_z",
    data=df_clean,
    model_name="Beta ~ All Total Scores"
)
summarize_model(model_beta_all, idata_beta_all, "Beta ~ All Total Scores")
plot_forest(idata_beta_all, 
            var_names=['DASS21_Total_z', 'AMI_Total_z', 'MFIS_Total_z', 
                       'Total_State_Score_z', 'Total_Trait_Score_z', 
                       'Total_OASIS_Score_z', 'Total_PHQ9_Score_z'],
            model_name="Beta ~ All Total Scores")

In [ ]:
# Model: k ~ all total scores
model_k_all, idata_k_all = fit_bayesian_model(
    formula="k_mean_z ~ DASS21_Total_z + AMI_Total_z + MFIS_Total_z + Total_State_Score_z + Total_Trait_Score_z + Total_OASIS_Score_z + Total_PHQ9_Score_z",
    data=df_clean,
    model_name="K ~ All Total Scores"
)
summarize_model(model_k_all, idata_k_all, "K ~ All Total Scores")
plot_forest(idata_k_all, 
            var_names=['DASS21_Total_z', 'AMI_Total_z', 'MFIS_Total_z', 
                       'Total_State_Score_z', 'Total_Trait_Score_z', 
                       'Total_OASIS_Score_z', 'Total_PHQ9_Score_z'],
            model_name="K ~ All Total Scores")

In [ ]:
# Model: z ~ all total scores
model_z_all, idata_z_all = fit_bayesian_model(
    formula="z_mean_z ~ DASS21_Total_z + AMI_Total_z + MFIS_Total_z + Total_State_Score_z + Total_Trait_Score_z + Total_OASIS_Score_z + Total_PHQ9_Score_z",
    data=df_clean,
    model_name="Z ~ All Total Scores"
)
summarize_model(model_z_all, idata_z_all, "Z ~ All Total Scores")
plot_forest(idata_z_all, 
            var_names=['DASS21_Total_z', 'AMI_Total_z', 'MFIS_Total_z', 
                       'Total_State_Score_z', 'Total_Trait_Score_z', 
                       'Total_OASIS_Score_z', 'Total_PHQ9_Score_z'],
            model_name="Z ~ All Total Scores")

---
# Part 4: Model Comparison

Compare models using LOO-CV.

In [ ]:
# Compare all beta models
beta_comparison_models = {
    'Beta ~ DASS21': beta_idatas['DASS21'],
    'Beta ~ AMI': beta_idatas['AMI'],
    'Beta ~ MFIS': beta_idatas['MFIS'],
    'Beta ~ State': beta_idatas['State'],
    'Beta ~ Trait': beta_idatas['Trait'],
    'Beta ~ OASIS': beta_idatas['OASIS'],
    'Beta ~ PHQ9': beta_idatas['PHQ9'],
    'Beta ~ All': idata_beta_all
}

beta_loo_comparison = az.compare(beta_comparison_models, ic='loo')
print("Beta Model Comparison (LOO):")
print(beta_loo_comparison)
print()

fig, ax = plt.subplots(figsize=(12, 8))
az.plot_compare(beta_loo_comparison, ax=ax)
ax.set_title('Beta Model Comparison (LOO-CV)')
plt.show()

In [ ]:
# Compare all k models
k_comparison_models = {
    'K ~ DASS21': k_idatas['DASS21'],
    'K ~ AMI': k_idatas['AMI'],
    'K ~ MFIS': k_idatas['MFIS'],
    'K ~ State': k_idatas['State'],
    'K ~ Trait': k_idatas['Trait'],
    'K ~ OASIS': k_idatas['OASIS'],
    'K ~ PHQ9': k_idatas['PHQ9'],
    'K ~ All': idata_k_all
}

k_loo_comparison = az.compare(k_comparison_models, ic='loo')
print("K Model Comparison (LOO):")
print(k_loo_comparison)
print()

fig, ax = plt.subplots(figsize=(12, 8))
az.plot_compare(k_loo_comparison, ax=ax)
ax.set_title('K Model Comparison (LOO-CV)')
plt.show()

In [ ]:
# Compare all z models
z_comparison_models = {
    'Z ~ DASS21': z_idatas['DASS21'],
    'Z ~ AMI': z_idatas['AMI'],
    'Z ~ MFIS': z_idatas['MFIS'],
    'Z ~ State': z_idatas['State'],
    'Z ~ Trait': z_idatas['Trait'],
    'Z ~ OASIS': z_idatas['OASIS'],
    'Z ~ PHQ9': z_idatas['PHQ9'],
    'Z ~ All': idata_z_all
}

z_loo_comparison = az.compare(z_comparison_models, ic='loo')
print("Z Model Comparison (LOO):")
print(z_loo_comparison)
print()

fig, ax = plt.subplots(figsize=(12, 8))
az.plot_compare(z_loo_comparison, ax=ax)
ax.set_title('Z Model Comparison (LOO-CV)')
plt.show()

---
# Part 5: Results Summary

Compile key findings from all models.

In [ ]:
def extract_coefficient_info(idata, predictor_name):
    """
    Extract coefficient estimates and credible intervals from inference data.
    """
    summary = az.summary(idata, hdi_prob=0.95)
    if predictor_name in summary.index:
        row = summary.loc[predictor_name]
        return {
            'mean': row['mean'],
            'sd': row['sd'],
            'hdi_3%': row['hdi_3%'],
            'hdi_97%': row['hdi_97%'],
            'credible_effect': 'Yes' if (row['hdi_3%'] > 0 or row['hdi_97%'] < 0) else 'No'
        }
    return None

# Create summary table for all total score models
total_score_results = []

# Mapping of score names to their standardized variable names and idata objects
score_mapping = {
    'DASS21_Total': ('DASS21_Total_z', 'DASS21'),
    'AMI_Total': ('AMI_Total_z', 'AMI'),
    'MFIS_Total': ('MFIS_Total_z', 'MFIS'),
    'Total_State_Score': ('Total_State_Score_z', 'State'),
    'Total_Trait_Score': ('Total_Trait_Score_z', 'Trait'),
    'Total_OASIS_Score': ('Total_OASIS_Score_z', 'OASIS'),
    'Total_PHQ9_Score': ('Total_PHQ9_Score_z', 'PHQ9')
}

# Beta models
for score_name, (var_name, key) in score_mapping.items():
    info = extract_coefficient_info(beta_idatas[key], var_name)
    if info:
        info['Parameter'] = 'Beta'
        info['Predictor'] = score_name
        total_score_results.append(info)

# K models
for score_name, (var_name, key) in score_mapping.items():
    info = extract_coefficient_info(k_idatas[key], var_name)
    if info:
        info['Parameter'] = 'K'
        info['Predictor'] = score_name
        total_score_results.append(info)

# Z models
for score_name, (var_name, key) in score_mapping.items():
    info = extract_coefficient_info(z_idatas[key], var_name)
    if info:
        info['Parameter'] = 'Z'
        info['Predictor'] = score_name
        total_score_results.append(info)

results_df = pd.DataFrame(total_score_results)
results_df = results_df[['Parameter', 'Predictor', 'mean', 'sd', 'hdi_3%', 'hdi_97%', 'credible_effect']]

print("=" * 100)
print("SUMMARY: All Total Score Models (Individual Predictors)")
print("=" * 100)
print(results_df.to_string(index=False))

In [ ]:
# Pivot table for easier viewing
pivot_mean = results_df.pivot(index='Predictor', columns='Parameter', values='mean').round(3)
pivot_credible = results_df.pivot(index='Predictor', columns='Parameter', values='credible_effect')

print("\n" + "=" * 60)
print("Coefficient Means by Parameter and Predictor")
print("=" * 60)
print(pivot_mean)

print("\n" + "=" * 60)
print("Credible Effects (95% HDI excludes 0)")
print("=" * 60)
print(pivot_credible)

In [ ]:
# Create a comprehensive visual summary
fig, axes = plt.subplots(3, 7, figsize=(24, 12))

params_list = ['Beta', 'K', 'Z']
predictors_list = list(score_mapping.keys())
idatas_dict = {
    'Beta': beta_idatas,
    'K': k_idatas,
    'Z': z_idatas
}

for i, param in enumerate(params_list):
    for j, (pred, (var_name, key)) in enumerate(score_mapping.items()):
        ax = axes[i, j]
        
        # Extract posterior samples for the predictor coefficient
        idata = idatas_dict[param][key]
        samples = idata.posterior[var_name].values.flatten()
        
        # Plot histogram
        ax.hist(samples, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
        ax.axvline(0, color='red', linestyle='--', linewidth=2)
        ax.axvline(np.mean(samples), color='black', linestyle='-', linewidth=2)
        
        # HDI
        hdi = az.hdi(samples, hdi_prob=0.95)
        ax.axvspan(hdi[0], hdi[1], alpha=0.2, color='green')
        
        # Determine if effect is credible
        credible = 'Yes' if (hdi[0] > 0 or hdi[1] < 0) else 'No'
        color = 'green' if credible == 'Yes' else 'gray'
        
        ax.set_xlabel('Coefficient', fontsize=8)
        ax.set_ylabel('Density', fontsize=8)
        ax.set_title(f'{param} ~ {pred.replace("_", " ")}\n(Credible: {credible})', 
                     fontsize=9, color=color)
        ax.tick_params(labelsize=7)

plt.suptitle('Posterior Distributions for All Total Score Models', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

---
# Part 6: Diagnostic Checks

In [ ]:
def run_diagnostics(idata, model_name):
    """
    Run and display diagnostic checks for a model.
    """
    print(f"\n{'='*60}")
    print(f"Diagnostics for: {model_name}")
    print(f"{'='*60}")
    
    # R-hat and ESS
    summary = az.summary(idata)
    print("\nR-hat values (should be < 1.01):")
    print(summary['r_hat'])
    
    print("\nEffective Sample Size (ESS):")
    print(summary[['ess_bulk', 'ess_tail']])
    
    # Trace plots
    az.plot_trace(idata)
    plt.suptitle(f'Trace Plots: {model_name}', y=1.02)
    plt.show()

# Run diagnostics for key models
run_diagnostics(beta_idatas['DASS21'], "Beta ~ DASS21 Total")
run_diagnostics(idata_beta_all, "Beta ~ All Total Scores")

---
# Part 7: Save Results

In [ ]:
# Save the summary results to CSV
results_df.to_csv('bayesian_regression_results_summary.csv', index=False)
print("Results saved to 'bayesian_regression_results_summary.csv'")

# Also save pivot tables
pivot_mean.to_csv('bayesian_regression_coefficients_pivot.csv')
print("Pivot table saved to 'bayesian_regression_coefficients_pivot.csv'")

---
# Part 8: Models with Covariates (Optional)

Include Age and Sex as covariates.

In [ ]:
# Encode Sex as numeric (if needed)
from scipy import stats

df_clean['Sex_numeric'] = df_clean['Sex'].map({'Male': 0, 'Female': 1})

# Standardize Age
df_clean['Age_z'] = stats.zscore(df_clean['Age'].dropna())

# Drop rows with missing covariates
df_cov = df_clean.dropna(subset=['Age_z', 'Sex_numeric'])
print(f"Subjects with complete covariate data: {len(df_cov)}")

In [ ]:
# Example: Model with covariates: beta ~ DASS21_Total + Age + Sex
model_beta_dass_cov, idata_beta_dass_cov = fit_bayesian_model(
    formula="beta_mean_z ~ DASS21_Total_z + Age_z + Sex_numeric",
    data=df_cov,
    model_name="Beta ~ DASS21 Total + Covariates"
)

summarize_model(model_beta_dass_cov, idata_beta_dass_cov, "Beta ~ DASS21 Total + Covariates")
plot_forest(idata_beta_dass_cov, 
            var_names=['DASS21_Total_z', 'Age_z', 'Sex_numeric'],
            model_name="Beta ~ DASS21 Total + Covariates")

In [ ]:
# Example: Model with covariates: beta ~ PHQ9 + Age + Sex
model_beta_phq9_cov, idata_beta_phq9_cov = fit_bayesian_model(
    formula="beta_mean_z ~ Total_PHQ9_Score_z + Age_z + Sex_numeric",
    data=df_cov,
    model_name="Beta ~ PHQ9 + Covariates"
)

summarize_model(model_beta_phq9_cov, idata_beta_phq9_cov, "Beta ~ PHQ9 + Covariates")
plot_forest(idata_beta_phq9_cov, 
            var_names=['Total_PHQ9_Score_z', 'Age_z', 'Sex_numeric'],
            model_name="Beta ~ PHQ9 + Covariates")

---
## Conclusions

This notebook provides a comprehensive Bayesian regression analysis examining how mental health questionnaire scores relate to computational model parameters (beta, k, z). 

### Mental Health Measures Analyzed:
1. **DASS21** - Depression, Anxiety, Stress (Total and Subscales)
2. **AMI** - Apathy Motivation Index (Total and Subscales)
3. **MFIS** - Modified Fatigue Impact Scale (Total and Subscales)
4. **State Anxiety** - STAI State subscale
5. **Trait Anxiety** - STAI Trait subscale
6. **OASIS** - Overall Anxiety Severity and Impairment Scale
7. **PHQ-9** - Patient Health Questionnaire for Depression

### Key Interpretation Guidelines:
- Effects where the 95% HDI excludes zero are considered credibly different from no effect
- Model comparison weights (LOO-CV) help decide which predictors are most informative
- Subscale models reveal which specific components drive relationships
- Combined models show unique contributions when controlling for other measures